In [1]:
# workflow.py

from typing import Dict, Any
from langgraph.graph import StateGraph, END
from schemas import WorkflowState
from nodes import (
    extract_context_node,
    work_package_classification_node,   # New import
    qdrant_insertion_node,
    action_parameter_extraction_node,
    action_execution_node,
    conditional_routing,  # will be overridden below
)

# NODE REGISTRY
nodes = {
    "extract_context": extract_context_node,
    "classify_work_packages": work_package_classification_node, 
    "extract_params": action_parameter_extraction_node,
    "execute_actions": action_execution_node,
    "qdrant_insertion": qdrant_insertion_node,
    "conditional_router": conditional_routing,  # keep for future use
    "end": END,
}

# WORKFLOW 
def create_workflow_graph():
    workflow = StateGraph(WorkflowState)

    # Nodes
    workflow.add_node("extract_context", nodes["extract_context"])
    workflow.add_node("classify_work_packages", nodes["classify_work_packages"])
    workflow.add_node("extract_params", nodes["extract_params"])
    workflow.add_node("qdrant_insertion", nodes["qdrant_insertion"])
    workflow.add_node("execute_actions", nodes["execute_actions"])

    # Entry point
    workflow.set_entry_point("extract_context")

    # Routing with updated logic
    workflow.add_conditional_edges(
    "extract_context",
    conditional_routing,   # just use your updated function
    {
        "classify_work_packages": "classify_work_packages",  # route here
        "END": nodes["end"],                                # or end
    }
)

    # Linear flow
    workflow.add_edge("classify_work_packages", "extract_params")
    workflow.add_edge("extract_params", "qdrant_insertion")
    workflow.add_edge("qdrant_insertion", "execute_actions")
    workflow.add_edge("execute_actions", nodes["end"])

    return workflow.compile()


ImportError: cannot import name 'WorkflowState' from 'schemas' (unknown location)

In [2]:
pip install langgraph


Note: you may need to restart the kernel to use updated packages.


In [53]:
from pydantic import BaseModel
from typing import List, Dict, Any

class WhatsAppState(TypedDict):
    whatsapp_messages: List[Dict]
    sites_string: str
    extraction_result: Optional[SiteIdExtraction]
    # Extracted data for next nodes
    site_ids: Optional[List[str]]
    site_names: Optional[List[str]]
    actions: Optional[List[str]]
    chat_id: Optional[List[str]]
    From: Optional[str]
    confidence: Optional[float]
    reasoning: Optional[str]
    # Context for parameter extraction
    context: Optional[Dict[str, Any]]
    error: Optional[str]


NameError: name 'TypedDict' is not defined

In [22]:
messages=[
  {
    "chat_id": "919398348980@s.whatsapp.net",
    "chat_name": "Me",
    "chat_type": "individual",
    "from_user": "919398348980@s.whatsapp.net",
    "message": "Heyy",
    "context": [],
    "timestamp": "2025-09-24T07:05:33+05:30"
  },
  {
    "chat_id": "919398348980@s.whatsapp.net",
    "chat_name": "Me",
    "chat_type": "individual",
    "from_user": "919398348980@s.whatsapp.net",
    "message": "Heyy",
    "context": [
      {
        "from": "919398348980@s.whatsapp.net",
        "to": "919398348980@s.whatsapp.net",
        "chat_name": "Me",
        "chat_type": "individual",
        "message_id": "ACDEAFCF601159DB012A3D090A64E8A7",
        "timestamp": "2025-09-24T07:05:33+05:30",
        "message": "Heyy"
      }
    ],
    "timestamp": "2025-09-24T07:05:33+05:30"
  }
]


In [ ]:
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field

class SiteIdExtraction(BaseModel):
    site_ids: Optional[List[Optional[str]]] = Field(default=None)
    site_names: Optional[List[Optional[str]]] = Field(default=None)
    actions: List[str] = Field(...)
    chat_id: List[str] = Field(...)
    From: str = Field(...)
    confidence: float = Field(...)
    reasoning: str = Field(...)

In [32]:
from config import OPENAI_CONFIG

In [33]:
import os
from langchain_openai import AzureChatOpenAI
llm = AzureChatOpenAI(
        azure_deployment=OPENAI_CONFIG['azure_deployment'],
        api_version=OPENAI_CONFIG['api_version'],
        azure_endpoint=OPENAI_CONFIG['azure_endpoint'],
        api_key=OPENAI_CONFIG['api_key'],
        temperature=0,
        max_tokens=None
    )

In [49]:
from langchain import PromptTemplate

context_prompt_template = PromptTemplate.from_template("""
You are an intelligent agent for infrastructure project management.

**PROCESS EACH NEW MESSAGE INDIVIDUALLY and return ONE action per message.**

### NEW MESSAGES:
{messages}

### Available sites: {sites}

### INSTRUCTIONS:

1. Extract for each new message:
   - One action per message
   - thread_id (use chat_id from WhatsApp)
   - message_id (use WhatsApp message_id if present, else null)
   - site info only if explicitly mentioned

2. Action rules:
   - `add_risk`: problems, delays, incidents — only if site is valid
   - `update_task`: task progress — only if site is valid
   - `update_risk`: mitigation completion — only if site is valid
   - `update`: general updates if site not mentioned or invalid
   - `ask_clarification`: ambiguous project-related message
   - `irrelevant`: casual, greetings, non-project content

### OUTPUT JSON FORMAT:

```json
{{
  "actions": ["action1", "action2"],
  "chat_id": ["chat1", "chat2"],
  "site_ids": ["site1", null],
  "site_names": ["Site Name 1", null],
  "From": "primary_sender",
  "confidence": 0.9,
  "reasoning": "Brief explanation of decisions"
}}
Arrays must match the number of NEW messages.

Use null for missing site info or message_id.

From, confidence, reasoning are single values for the batch.
""")

In [50]:
from pydantic import BaseModel, Field
from langchain.output_parsers import PydanticOutputParser



In [51]:
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field
from langchain.schema import HumanMessage


class SiteIdExtraction(BaseModel):
    site_ids: Optional[List[Optional[str]]] = Field(default=None)
    site_names: Optional[List[Optional[str]]] = Field(default=None)
    actions: List[str] = Field(...)
    chat_id: List[str] = Field(...)
    From: str = Field(...)
    confidence: float = Field(...)
    reasoning: str = Field(...)

parser = PydanticOutputParser(return_id=False, pydantic_object=SiteIdExtraction)


def extract_whatsapp_context(
    whatsapp_messages: List[Dict],
    sites_string: str,
    llm,
) -> SiteIdExtraction:
    """
    Simple function to extract actions, site info, and chat_id from WhatsApp messages
    using an LLM. Directly sends messages to LLM with context.
    """

    # Prepare prompt for LLM
    prompt = context_prompt_template.format(
        messages=whatsapp_messages,
        sites=sites_string,
        format_instructions=parser.get_format_instructions()  # if your parser has instructions, include here
    )

    # Call LLM
    response = llm.invoke([HumanMessage(content=prompt)])
    result = parser.parse(response.content)  # structured output parser


    return SiteIdExtraction(
        site_ids=result.site_ids,
        site_names=result.site_names,
        actions=result.actions,
        chat_id=result.chat_id,
        From=result.From,
        confidence=result.confidence,
        reasoning=result.reasoning
    )


In [54]:
def whatsapp_context_extraction_node(state: WhatsAppState, llm) -> WhatsAppState:
    """
    Node function to extract WhatsApp context and add results to state.
    """
    try:
        
        # Get data from state
        whatsapp_messages = state.get("whatsapp_messages", [])
        sites_string = state.get("sites_string", "")
        
        if not whatsapp_messages:
            return {
                **state,
                "error": "No WhatsApp messages provided",
                "extraction_result": None
            }
        
        # Extract context
        extraction_result = extract_whatsapp_context(
            whatsapp_messages=whatsapp_messages,
            sites_string=sites_string,
            llm=llm
        )
        
        # Create context for next nodes (parameter extraction)
        context = {
            "reasoning": extraction_result.reasoning,
            "site_ids": extraction_result.site_ids,
            "site_names": extraction_result.site_names,
            "chat_id": extraction_result.chat_id,
            "From": extraction_result.From,
            "new_messages": whatsapp_messages,  # For parameter extraction templates
            "context_messages_content": "",  # Add if you have context messages
        }
        
        # Update state with results - individual fields for easy access
        return {
            **state,
            "extraction_result": extraction_result,
            "site_ids": extraction_result.site_ids,
            "site_names": extraction_result.site_names, 
            "actions": extraction_result.actions,
            "chat_id": extraction_result.chat_id,
            "From": extraction_result.From,
            "confidence": extraction_result.confidence,
            "reasoning": extraction_result.reasoning,
            "context": context,  # Ready for parameter extraction
            "error": None
        }
        
    except Exception as e:
        return {
            **state,
            "error": f"Context extraction failed: {str(e)}",
            "extraction_result": None
        }

NameError: name 'WhatsAppState' is not defined

In [52]:
sites_string = "Site A, Site B, Site C"  # list of available sites

result = extract_whatsapp_context(messages, sites_string, llm)

# Print structured output
print(result.model_dump_json(indent=2))

{
  "site_ids": [
    null,
    null
  ],
  "site_names": [
    null,
    null
  ],
  "actions": [
    "irrelevant",
    "irrelevant"
  ],
  "chat_id": [
    "919398348980@s.whatsapp.net",
    "919398348980@s.whatsapp.net"
  ],
  "From": "919398348980@s.whatsapp.net",
  "confidence": 0.95,
  "reasoning": "Both messages contain only the casual greeting 'Heyy' with no project-related content or site mention, so they are classified as irrelevant."
}


In [ ]:
# nodes.py
from typing import Dict, Any
from utils import get_enhanced_task_context_from_message, extract_parameters_with_llm, get_task_context_for_llm, get_packages_by_site, get_assets_with_cwps_and_iwps, get_iwp_context_for_llm
import logging

logger = logging.getLogger(__name__)

def extract_context_node(state: Dict[str, Any]) -> Dict[str, Any]:
    # Extracts context from emails batch
    ...

def work_package_classification_node(state: Dict[str, Any]) -> Dict[str, Any]:
    # Classifies actions per CWPs/IWPs
    ...

def action_parameter_extraction_node(state: Dict[str, Any]) -> Dict[str, Any]:
    # Extracts parameters per unique action
    ...

async def action_execution_node(state: Dict[str, Any]) -> Dict[str, Any]:
    # Executes actions (add_risk, update_task, ask_clarification)
    ...

def qdrant_insertion_node(state: Dict[str, Any]) -> Dict[str, Any]:
    # Inserts message embeddings into Qdrant
    ...

def conditional_routing(state: Dict[str, Any]) -> str:
    # Determines next node based on extracted actions
    ...
